# S0.1 — CMIP6 Model Selection & Download Plan / 模型选择与下载清单

Based on the S0 compatibility analysis, this notebook:

1. **Family deduplication** — group models by shared physical core, keep one representative per family
2. **Model scoring** — rank models within each family by data completeness
3. **Member selection** — choose the single best ensemble member per model
4. **Download manifest** — produce a CSV listing every (model, variable, table, member) to download

**中文说明：** 基于S0兼容性分析的结果，本 Notebook 完成模型选择：按模型家族去重（避免同一物理核心的模型重复计入），在每个家族内按数据完整度打分选出代表模型，为每个模型选择最优 member，最终生成下载清单 CSV。

## 1. Configuration & data loading / 配置与数据加载

**中文说明：** 加载 ESGF 目录和 S0 输出，定义核心变量、Context 变量和模型家族规则。

In [1]:
from __future__ import annotations

import re
from collections import defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display


def locate_case_dir() -> Path:
    for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if d.name == "caseA":
            return d
        nested = d / "case" / "caseA"
        if nested.is_dir():
            return nested
    raise FileNotFoundError("Could not locate case/caseA")


CASE_DIR = locate_case_dir()
CATALOG_JSONL = CASE_DIR / "CMIP_information" / "esgf_historical_model_variable_docs.jsonl"
S0_DIR = CASE_DIR / "output" / "S0"
OUT_DIR = CASE_DIR / "output" / "S0.1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Variable groups ──
CORE = [("pr", "Amon"), ("evspsbl", "Amon"), ("mrro", "Lmon")]

CONTEXT_VARS = [
    ("tas",         "Amon"),   # near-surface air temperature
    ("rsds",        "Amon"),   # surface downwelling shortwave
    ("rsus",        "Amon"),   # surface upwelling shortwave
    ("rlds",        "Amon"),   # surface downwelling longwave
    ("rlus",        "Amon"),   # surface upwelling longwave
    ("hfls",        "Amon"),   # surface latent heat flux
    ("hfss",        "Amon"),   # surface sensible heat flux
    ("mrso",        "Lmon"),   # total soil moisture
    ("lai",         "Lmon"),   # leaf area index
    ("mrsos",       "Lmon"),   # surface soil moisture (top ~10 cm)
    ("mrros",       "Lmon"),   # surface runoff
    ("prsn",        "Amon"),   # snowfall flux
    ("snw",         "LImon"),  # surface snow amount / snow water equivalent
    ("evspsblsoi",  "Lmon"),   # soil evaporation
    ("tran",        "Lmon"),   # transpiration
]

FIXED_FIELDS = [("sftlf", "fx"), ("areacella", "fx")]

TARGET_VARS = CORE + CONTEXT_VARS

# ── Time period thresholds (same as S0) ──
ANALYSIS_START_YEAR = 1985
PASS_STOP = pd.Timestamp("2014-12-01", tz="UTC")
FLAG_STOP = pd.Timestamp("2012-01-01", tz="UTC")

# ── Family rules ──
# (family_name, [prefix, ...]). A model matches if it starts with any prefix.
# Models not matching any rule become a singleton family.
# Split where atmosphere/ocean model differs; keep together when only
# ESM components, resolution, or chemistry schemes differ.
FAMILY_RULES = [
    ("ACCESS-CM2",     ["ACCESS-CM2"]),
    ("ACCESS-ESM1",    ["ACCESS-ESM1"]),
    ("AWI",            ["AWI"]),
    ("BCC",            ["BCC"]),
    ("CAMS",           ["CAMS"]),
    ("CAS",            ["CAS"]),
    ("CESM2",          ["CESM2"]),
    ("CIESM",          ["CIESM"]),
    ("CMCC",           ["CMCC"]),
    ("CNRM",           ["CNRM"]),
    ("CanESM",         ["CanESM"]),
    ("E3SM",           ["E3SM"]),
    ("EC-Earth3",      ["EC-Earth3"]),
    ("FGOALS-f3",      ["FGOALS-f3"]),
    ("FGOALS-g3",      ["FGOALS-g3"]),
    ("FIO",            ["FIO"]),
    ("GFDL",           ["GFDL"]),
    ("GISS-E2-1",      ["GISS-E2-1"]),
    ("GISS-E2-2",      ["GISS-E2-2"]),
    ("HadGEM3-UKESM",  ["HadGEM3", "UKESM"]),
    ("INM",            ["INM"]),
    ("IPSL-CM5A2",     ["IPSL-CM5A2"]),
    ("IPSL-CM6A",      ["IPSL-CM6A"]),
    ("KACE",           ["KACE"]),
    ("KIOST",          ["KIOST"]),
    ("MCM-UA",         ["MCM"]),
    ("MIROC-ES2",      ["MIROC-ES2"]),
    ("MIROC6",         ["MIROC6"]),
    ("MPI",            ["MPI"]),
    ("MRI",            ["MRI"]),
    ("NESM",           ["NESM"]),
    ("NorCPM",         ["NorCPM"]),
    ("NorESM",         ["NorESM"]),
    ("SAM0",           ["SAM0"]),
    ("TaiESM",         ["TaiESM"]),
]

print("Core variables:", CORE)
print("Context variables:", CONTEXT_VARS)
print(f"Target variables (core + context): {len(TARGET_VARS)}")
print(f"Family rules: {len(FAMILY_RULES)} families defined")

Core variables: [('pr', 'Amon'), ('evspsbl', 'Amon'), ('mrro', 'Lmon')]
Context variables: [('tas', 'Amon'), ('rsds', 'Amon'), ('rsus', 'Amon'), ('rlds', 'Amon'), ('rlus', 'Amon'), ('hfls', 'Amon'), ('hfss', 'Amon'), ('mrso', 'Lmon'), ('lai', 'Lmon'), ('mrsos', 'Lmon'), ('mrros', 'Lmon'), ('prsn', 'Amon'), ('snw', 'LImon'), ('evspsblsoi', 'Lmon'), ('tran', 'Lmon')]
Target variables (core + context): 18
Family rules: 35 families defined


In [2]:
# ── Reload ESGF catalog (same logic as S0) ──
raw = pd.read_json(CATALOG_JSONL, lines=True)
raw = raw.dropna(subset=["source_id", "variable_id", "table_id", "member_id"])
raw["replica"] = raw["replica"].fillna(False).astype(bool)
raw = raw.sort_values(["master_id", "replica"])
docs = raw.drop_duplicates("master_id", keep="first").copy()


def parse_time(v):
    if v is None or v == "" or (isinstance(v, float) and pd.isna(v)):
        return pd.NaT
    return pd.to_datetime(v, utc=True, errors="coerce")


docs["start"] = docs["datetime_start"].map(parse_time)
docs["stop"] = docs["datetime_stop"].map(parse_time)

ALL_MODELS = sorted(docs["source_id"].unique())
member_sets = docs.groupby(["source_id", "variable_id", "table_id"])["member_id"].apply(set).to_dict()
grid_index = (
    docs.dropna(subset=["grid_label"])
    .groupby(["source_id", "variable_id", "table_id", "member_id"])["grid_label"]
    .apply(set)
    .to_dict()
)

# ── Time index: (model, variable, table, member) -> (earliest_start, latest_stop) ──
_time_agg = docs.groupby(["source_id", "variable_id", "table_id", "member_id"]).agg(
    start=("start", "min"), stop=("stop", "max")
)
time_index = {k: (row.start, row.stop) for k, row in _time_agg.iterrows()}


def period_status(start, stop):
    if pd.isna(start) or start.year > ANALYSIS_START_YEAR:
        return "Fail"
    if pd.isna(stop):
        return "Flag"
    if stop >= PASS_STOP:
        return "Pass"
    if stop >= FLAG_STOP:
        return "Flag"
    return "Fail"


# ── Load S0 core compatibility ──
summary_core = pd.read_csv(S0_DIR / "S0_core_compatibility.csv")
for col in ["sftlf_fx", "areacella_fx", "sftgif_any", "same_grid"]:
    if col in summary_core.columns:
        summary_core[col] = summary_core[col].fillna(False)
        summary_core[col] = summary_core[col].map(
            lambda x: x if isinstance(x, bool) else str(x).strip().lower() == "true"
        )
summary_core = summary_core.set_index("model")

# ── mrro member lookup ──
mrro_members = {m: member_sets.get((m, "mrro", "Lmon"), set()) for m in ALL_MODELS}
models_with_mrro = sorted(m for m, s in mrro_members.items() if s)

print(f"Catalog: {len(docs):,} records, {len(ALL_MODELS)} models")
print(f"Models with mrro: {len(models_with_mrro)}")
print(f"Time index entries: {len(time_index):,}")
print(f"S0 core summary loaded: {len(summary_core)} rows")

Catalog: 273,607 records, 75 models
Models with mrro: 68
Time index entries: 266,485
S0 core summary loaded: 75 rows


/var/folders/y6/36h2pwl51ql85zwkh35jmm1r0000gp/T/ipykernel_38943/3982864354.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  summary_core[col] = summary_core[col].fillna(False)


## 2. Family assignment / 模型家族分配

同一家族的模型共享底层大气/海洋/陆面耦合模式，差异仅在分辨率、化学方案或 ESM 组件。
例如 UKESM1 和 HadGEM3 共享物理核心，CESM2 的各变体（FV2、WACCM 等）属于同一家族。

**中文说明：** 将 75 个模型按共享的物理核心分配到家族，避免同一家族多个模型重复计入分析。

In [3]:
def assign_family(model: str) -> str:
    for family_name, prefixes in FAMILY_RULES:
        if any(model.startswith(p) for p in prefixes):
            return family_name
    return model

model_family = {m: assign_family(m) for m in ALL_MODELS}

family_df = pd.DataFrame([
    {"model": m, "family": f, "has_mrro": m in models_with_mrro}
    for m, f in sorted(model_family.items(), key=lambda x: (x[1], x[0]))
])

family_sizes = family_df.groupby("family").agg(
    n_models=("model", "count"),
    n_with_mrro=("has_mrro", "sum"),
    models=("model", lambda x: ", ".join(x)),
).reset_index().sort_values("n_models", ascending=False)

print(f"Total models: {len(ALL_MODELS)}")
print(f"Families: {family_df['family'].nunique()}")
print(f"Families with \u22651 mrro model: {family_df[family_df['has_mrro']]['family'].nunique()}")
print()
display(family_sizes)


Total models: 75
Families: 38
Families with ≥1 mrro model: 35



,family,n_models,n_with_mrro,models
12,EC-Earth3,8,5,"EC-Earth3, EC-Earth3-AerChem, EC-Earth3-CC, EC..."
11,E3SM,6,6,"E3SM-1-0, E3SM-1-1, E3SM-1-1-ECA, E3SM-2-0, E3..."
6,CESM2,4,4,"CESM2, CESM2-FV2, CESM2-WACCM, CESM2-WACCM-FV2"
20,HadGEM3-UKESM,4,4,"HadGEM3-GC31-LL, HadGEM3-GC31-MM, UKESM1-0-LL,..."
9,CNRM,3,3,"CNRM-CM6-1, CNRM-CM6-1-HR, CNRM-ESM2-1"
2,AWI,3,2,"AWI-CM-1-1-MR, AWI-ESM-1-1-LR, AWI-ESM-1-REcoM"
17,GISS-E2-1,3,3,"GISS-E2-1-G, GISS-E2-1-G-CC, GISS-E2-1-H"
31,MPI,3,3,"MPI-ESM-1-2-HAM, MPI-ESM1-2-HR, MPI-ESM1-2-LR"
8,CMCC,3,3,"CMCC-CM2-HR4, CMCC-CM2-SR5, CMCC-ESM2"
10,CanESM,3,3,"CanESM5, CanESM5-1, CanESM5-CanOE"


## 3. Model scoring & family representative selection / 模型打分与家族代表选择

Scoring priority (highest first):
1. **Core time status**: Pass (4) > Flag (3) > Fail (2) > P+R only (1) > none (0)
2. **Fixed fields**: sftlf + areacella both available (2) > one (1) > neither (0)
3. **Core same grid**: same (1) > different or N/A (0)
4. **Target variable coverage**: count of core + context variables with mrro member overlap
5. **Model name**: alphabetical tiebreaker (ensures stable ordering)

Within each family, the highest-scoring model with mrro is selected as the representative.

**中文说明：** 在每个家族内，按以下优先级对模型打分：(1) 核心三变量时间覆盖状态（5档：Pass > Flag > Fail > 仅P+R > 无交集）；(2) 固定场可用性；(3) 核心变量网格一致性；(4) 目标变量覆盖数；(5) 模型名称字母序（保证排序稳定）。每个家族取得分最高的模型作为代表。

In [4]:
score_rows = []
for model in models_with_mrro:
    if model in summary_core.index:
        ci = summary_core.loc[model]
        n_common = int(ci["n_common"])
        time_status = str(ci.get("time_status", "—"))
        sftlf = bool(ci.get("sftlf_fx", False))
        areacella = bool(ci.get("areacella_fx", False))
        same_grid = bool(ci.get("same_grid", False))
    else:
        n_common, time_status, sftlf, areacella, same_grid = 0, "—", False, False, False

    # P+R intersection check
    pr_m = member_sets.get((model, "pr", "Amon"), set())
    mrro_m = mrro_members[model]
    has_pr_r = bool(pr_m & mrro_m)

    # 5-tier core_rank: Pass(4) > Flag(3) > Fail(2) > P+R-only(1) > none(0)
    if time_status == "Pass":
        core_rank = 4
    elif time_status == "Flag":
        core_rank = 3
    elif n_common > 0:
        core_rank = 2
    elif has_pr_r:
        core_rank = 1
    else:
        core_rank = 0

    fixed_rank = int(sftlf) + int(areacella)
    grid_rank = int(same_grid)

    n_target = sum(
        1 for v, t in TARGET_VARS
        if member_sets.get((model, v, t), set()) & mrro_m
    )

    score_rows.append({
        "model": model,
        "family": model_family[model],
        "core_rank": core_rank,
        "fixed_rank": fixed_rank,
        "grid_rank": grid_rank,
        "n_target_vars": n_target,
        "core_time": time_status,
        "has_pr_r": has_pr_r,
        "sftlf": sftlf,
        "areacella": areacella,
        "same_grid": same_grid,
    })

score_df = pd.DataFrame(score_rows).sort_values(
    ["family", "core_rank", "fixed_rank", "grid_rank", "n_target_vars", "model"],
    ascending=[True, False, False, False, False, True],
)

selected = score_df.groupby("family").first().reset_index()
selected = selected.sort_values(
    ["core_rank", "fixed_rank", "n_target_vars", "model"],
    ascending=[False, False, False, True],
)

print(f"Models scored: {len(score_df)}")
print(f"Families with mrro: {score_df['family'].nunique()}")
print(f"Selected representatives: {len(selected)}")
print()
print("=== Selected family representatives ===")
display(selected[[
    "family", "model", "core_rank", "fixed_rank", "grid_rank",
    "n_target_vars", "core_time", "has_pr_r", "sftlf", "areacella", "same_grid",
]])

print()
print("=== Full scoring table (all 68 models) ===")
display(score_df[[
    "family", "model", "core_rank", "fixed_rank", "grid_rank",
    "n_target_vars", "core_time", "has_pr_r",
]])

Models scored: 68
Families with mrro: 35
Selected representatives: 35

=== Selected family representatives ===


,family,model,core_rank,fixed_rank,grid_rank,n_target_vars,core_time,has_pr_r,sftlf,areacella,same_grid
6,CESM2,CESM2,4,2,1,18,Pass,True,True,True,True
9,CNRM,CNRM-CM6-1,4,2,1,18,Pass,True,True,True,True
10,CanESM,CanESM5,4,2,1,18,Pass,True,True,True,True
11,E3SM,E3SM-1-0,4,2,1,18,Pass,True,True,True,True
16,GFDL,GFDL-CM4,4,2,1,18,Pass,True,True,True,True
24,IPSL-CM6A,IPSL-CM6A-LR,4,2,1,18,Pass,True,True,True,True
28,MIROC-ES2,MIROC-ES2H,4,2,1,18,Pass,True,True,True,True
31,MRI,MRI-ESM2-0,4,2,1,18,Pass,True,True,True,True
33,SAM0,SAM0-UNICON,4,2,1,18,Pass,True,True,True,True
34,TaiESM,TaiESM1,4,2,1,18,Pass,True,True,True,True



=== Full scoring table (all 68 models) ===


,family,model,core_rank,fixed_rank,grid_rank,n_target_vars,core_time,has_pr_r
0,ACCESS-CM2,ACCESS-CM2,3,2,1,16,Flag,True
1,ACCESS-ESM1,ACCESS-ESM1-5,3,2,1,17,Flag,True
2,AWI,AWI-ESM-1-1-LR,3,2,1,17,Flag,True
3,AWI,AWI-ESM-1-REcoM,0,0,0,3,—,False
4,BCC,BCC-CSM2-MR,4,0,1,18,Pass,True
...,...,...,...,...,...,...,...,...
61,MRI,MRI-ESM2-0,4,2,1,18,Pass,True
62,NorESM,NorESM2-LM,3,2,1,18,Flag,True
63,NorESM,NorESM2-MM,3,2,1,18,Flag,True
64,SAM0,SAM0-UNICON,4,2,1,18,Pass,True


## 4. Member selection / Member 选择

For each selected model, choose the best ensemble member:

1. **Candidate pool**: core common members (pr ∩ evspsbl ∩ mrro) if available, else P+R common members (pr ∩ mrro), else any mrro member
2. **Time filter**: drop candidates where any pool-relevant variable has Fail time status (core pool checks all 3 core vars; P+R pool checks pr + mrro). If all candidates fail, keep the original set as fallback.
3. **Scoring**: count of target variables (core + context) available on that member
4. **Tiebreak**: smallest r/i/p/f indices

**中文说明：** 为每个选出的模型选择最优 member。候选池优先取核心三变量交集 member，不存在则取 P+R 交集，再不行则取任何有 mrro 的 member。先过滤掉时间覆盖 Fail 的 member（如全部 Fail 则保留原候选集作为兜底）。在剩余候选中，选择覆盖最多目标变量的 member，平手按 r/i/p/f 编号从小到大排序。

In [5]:
def parse_ripf(member_id: str) -> tuple:
    nums = re.findall(r"\d+", member_id)
    return tuple(int(x) for x in nums) if nums else (999,)


def member_period(model, var, tbl, member):
    ti = time_index.get((model, var, tbl, member))
    if ti is None:
        return "Fail"
    return period_status(ti[0], ti[1])


member_rows = []
for _, row in selected.iterrows():
    model = row["model"]
    family = row["family"]
    mrro_m = mrro_members[model]
    pr_m = member_sets.get((model, "pr", "Amon"), set())
    et_m = member_sets.get((model, "evspsbl", "Amon"), set())

    core_common = pr_m & et_m & mrro_m
    pr_r_common = pr_m & mrro_m

    if core_common:
        candidates = core_common
        pool = "core"
    elif pr_r_common:
        candidates = pr_r_common
        pool = "P+R"
    else:
        candidates = mrro_m
        pool = "mrro-only"

    # ── Time filter: drop members where pool-relevant vars have Fail ──
    if pool == "core":
        time_ok = {
            m for m in candidates
            if all(member_period(model, v, t, m) != "Fail" for v, t in CORE)
        }
    elif pool == "P+R":
        time_ok = {
            m for m in candidates
            if (member_period(model, "pr", "Amon", m) != "Fail"
                and member_period(model, "mrro", "Lmon", m) != "Fail")
        }
    else:
        time_ok = {
            m for m in candidates
            if member_period(model, "mrro", "Lmon", m) != "Fail"
        }

    filtered = time_ok if time_ok else candidates
    n_dropped = len(candidates) - len(filtered)

    # ── Score by target variable coverage, tiebreak by ripf ──
    best_member = None
    best_key = (-1, (999,))
    for member in filtered:
        n_vars = sum(
            1 for v, t in TARGET_VARS
            if member in member_sets.get((model, v, t), set())
        )
        ripf = parse_ripf(member)
        key = (n_vars, tuple(-x for x in ripf))
        if key > best_key:
            best_key = key
            best_member = member

    available = [f"{v}/{t}" for v, t in TARGET_VARS
                 if best_member in member_sets.get((model, v, t), set())]
    missing = [f"{v}/{t}" for v, t in TARGET_VARS
               if best_member not in member_sets.get((model, v, t), set())]

    member_rows.append({
        "model": model,
        "family": family,
        "member_id": best_member,
        "pool": pool,
        "n_candidates": len(candidates),
        "n_time_dropped": n_dropped,
        "n_target_available": len(available),
        "n_target_missing": len(missing),
        "available": ", ".join(available),
        "missing": ", ".join(missing),
    })

member_df = pd.DataFrame(member_rows)

print(f"=== Member selection results ({len(member_df)} models) ===")
print(f"  From core pool: {(member_df['pool'] == 'core').sum()}")
print(f"  From P+R pool: {(member_df['pool'] == 'P+R').sum()}")
print(f"  From mrro-only pool: {(member_df['pool'] == 'mrro-only').sum()}")
print(f"  Members dropped by time filter: {member_df['n_time_dropped'].sum()}")
print(f"\n  Full target coverage ({len(TARGET_VARS)}/{len(TARGET_VARS)}): "
      f"{(member_df['n_target_available'] == len(TARGET_VARS)).sum()}")
print(f"  Missing ≥1 target var: {(member_df['n_target_missing'] > 0).sum()}")
print()
display(member_df[[
    "family", "model", "member_id", "pool", "n_candidates",
    "n_time_dropped", "n_target_available", "n_target_missing", "missing",
]])

=== Member selection results (35 models) ===
  From core pool: 33
  From P+R pool: 2
  From mrro-only pool: 0
  Members dropped by time filter: 33

  Full target coverage (18/18): 16
  Missing ≥1 target var: 19



,family,model,member_id,pool,n_candidates,n_time_dropped,n_target_available,n_target_missing,missing
0,CESM2,CESM2,r1i1p1f1,core,11,1,18,0,
1,CNRM,CNRM-CM6-1,r1i1p1f2,core,29,0,18,0,
2,CanESM,CanESM5,r1i1p1f1,core,65,0,18,0,
3,E3SM,E3SM-1-0,r1i1p1f1,core,25,0,18,0,
4,GFDL,GFDL-CM4,r1i1p1f1,core,1,0,18,0,
5,IPSL-CM6A,IPSL-CM6A-LR,r1i1p1f1,core,33,0,18,0,
6,MIROC-ES2,MIROC-ES2H,r1i1p4f2,core,3,0,18,0,
7,MRI,MRI-ESM2-0,r1i2p1f1,core,12,0,18,0,
8,SAM0,SAM0-UNICON,r1i1p1f1,core,1,0,18,0,
9,TaiESM,TaiESM1,r1i1p1f1,core,2,0,18,0,


## 5. Download manifest / 下载清单

For each selected model × member, list every dataset to download:
- **core**: pr/Amon, evspsbl/Amon, mrro/Lmon — even if missing, listed for completeness
- **fixed**: sftlf/fx, areacella/fx — any member suffices
- **context**: target context variables available on the selected member

**中文说明：** 为每个选出的模型和 member 生成下载清单。包括核心变量（即使缺失也列出）、固定场（不区分 member）和该 member 上可用的 Context 变量。输出为 CSV 文件，供 S1 下载脚本读取。

In [6]:
def get_grid(model, var, tbl, member):
    g = grid_index.get((model, var, tbl, member), set())
    return ", ".join(sorted(g)) if g else ""


manifest_rows = []
for _, row in member_df.iterrows():
    model = row["model"]
    family = row["family"]
    member = row["member_id"]

    # Core variables
    for var, tbl in CORE:
        avail = member in member_sets.get((model, var, tbl), set())
        manifest_rows.append({
            "model": model,
            "family": family,
            "member_id": member,
            "variable": var,
            "table_id": tbl,
            "grid_label": get_grid(model, var, tbl, member) if avail else "",
            "category": "core",
            "status": "available" if avail else "missing",
        })

    # Fixed fields (any member — pick smallest ripf)
    for var, tbl in FIXED_FIELDS:
        all_m = member_sets.get((model, var, tbl), set())
        if all_m:
            fx_member = min(all_m, key=parse_ripf)
            manifest_rows.append({
                "model": model,
                "family": family,
                "member_id": fx_member,
                "variable": var,
                "table_id": tbl,
                "grid_label": get_grid(model, var, tbl, fx_member),
                "category": "fixed",
                "status": "available",
            })
        else:
            manifest_rows.append({
                "model": model,
                "family": family,
                "member_id": "",
                "variable": var,
                "table_id": tbl,
                "grid_label": "",
                "category": "fixed",
                "status": "missing",
            })

    # Context variables (on selected member only)
    for var, tbl in CONTEXT_VARS:
        avail = member in member_sets.get((model, var, tbl), set())
        manifest_rows.append({
            "model": model,
            "family": family,
            "member_id": member if avail else "",
            "variable": var,
            "table_id": tbl,
            "grid_label": get_grid(model, var, tbl, member) if avail else "",
            "category": "context",
            "status": "available" if avail else "not on member",
        })

manifest_df = pd.DataFrame(manifest_rows)
n_avail = (manifest_df["status"] == "available").sum()
n_miss = (manifest_df["status"] != "available").sum()

print("=== Download manifest ===")
print(f"  Models: {manifest_df['model'].nunique()}")
print(f"  Total entries: {len(manifest_df)} ({n_avail} available, {n_miss} missing/unavailable)")
print(f"  By category:")
for cat in ["core", "fixed", "context"]:
    sub = manifest_df[manifest_df["category"] == cat]
    print(f"    {cat}: {(sub['status'] == 'available').sum()} / {len(sub)} available")
print()

# Downloadable rows only
download_df = manifest_df[manifest_df["status"] == "available"].copy()

model_summary = download_df.groupby(["model", "family"]).agg(
    n_datasets=("variable", "count"),
    variables=("variable", lambda x: ", ".join(x)),
).reset_index().sort_values("n_datasets", ascending=False)

print(f"=== Downloadable datasets per model ({len(download_df)} total) ===")
display(model_summary)
print()
print("=== Full manifest (first 40 rows) ===")
display(manifest_df.head(40))

=== Download manifest ===
  Models: 35
  Total entries: 700 (629 available, 71 missing/unavailable)
  By category:
    core: 103 / 105 available
    fixed: 47 / 70 available
    context: 479 / 525 available

=== Downloadable datasets per model (629 total) ===


,model,family,n_datasets,variables
9,CNRM-CM6-1,CNRM,20,"pr, evspsbl, mrro, sftlf, areacella, tas, rsds..."
30,MRI-ESM2-0,MRI,20,"pr, evspsbl, mrro, sftlf, areacella, tas, rsds..."
16,GFDL-CM4,GFDL,20,"pr, evspsbl, mrro, sftlf, areacella, tas, rsds..."
27,MIROC-ES2H,MIROC-ES2,20,"pr, evspsbl, mrro, sftlf, areacella, tas, rsds..."
11,E3SM-1-0,E3SM,20,"pr, evspsbl, mrro, sftlf, areacella, tas, rsds..."
10,CanESM5,CanESM,20,"pr, evspsbl, mrro, sftlf, areacella, tas, rsds..."
19,GISS-E3-G,GISS-E3-G,20,"pr, evspsbl, mrro, sftlf, areacella, tas, rsds..."
23,IPSL-CM6A-LR,IPSL-CM6A,20,"pr, evspsbl, mrro, sftlf, areacella, tas, rsds..."
6,CESM2,CESM2,20,"pr, evspsbl, mrro, sftlf, areacella, tas, rsds..."
31,NorESM2-LM,NorESM,20,"pr, evspsbl, mrro, sftlf, areacella, tas, rsds..."



=== Full manifest (first 40 rows) ===


,model,family,member_id,variable,table_id,grid_label,category,status
0,CESM2,CESM2,r1i1p1f1,pr,Amon,gn,core,available
1,CESM2,CESM2,r1i1p1f1,evspsbl,Amon,gn,core,available
2,CESM2,CESM2,r1i1p1f1,mrro,Lmon,gn,core,available
3,CESM2,CESM2,r1i1p1f1,sftlf,fx,gn,fixed,available
4,CESM2,CESM2,r1i1p1f1,areacella,fx,gn,fixed,available
5,CESM2,CESM2,r1i1p1f1,tas,Amon,gn,context,available
6,CESM2,CESM2,r1i1p1f1,rsds,Amon,gn,context,available
7,CESM2,CESM2,r1i1p1f1,rsus,Amon,gn,context,available
8,CESM2,CESM2,r1i1p1f1,rlds,Amon,gn,context,available
9,CESM2,CESM2,r1i1p1f1,rlus,Amon,gn,context,available


In [7]:
manifest_df.to_csv(OUT_DIR / "S0.1_download_manifest.csv", index=False)
download_df = manifest_df[manifest_df["status"] == "available"]
download_df.to_csv(OUT_DIR / "S0.1_download_available.csv", index=False)
member_df.to_csv(OUT_DIR / "S0.1_selected_models.csv", index=False)
score_df.to_csv(OUT_DIR / "S0.1_model_scores.csv", index=False)

print("Saved to", OUT_DIR)
for f in sorted(OUT_DIR.glob("S0.1_*.csv")):
    print(" ", f.name)
print()
print(f"\nNext step: use S0.1_download_available.csv as input to S1 download script.")
print(f"Total datasets to download: {len(download_df)}")


Saved to /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S0.1
  S0.1_download_available.csv
  S0.1_download_manifest.csv
  S0.1_model_scores.csv
  S0.1_selected_models.csv


Next step: use S0.1_download_available.csv as input to S1 download script.
Total datasets to download: 629
